Import packages:

In [1]:
import numpy as np
import numpy.random
import scipy
import scipy.linalg #this contains the tools needed to solve sytems of equations
import pprint
pp = pprint.PrettyPrinter(indent=4)

## **LIBRARY'S SOLVER + LU DECOMPOSITION + PIVOTING**




We want to solve the system Ax = b

In [ ]:
A = np.array( [ [1,2,3], [0,5,6], [7,0,9] ])
b = np.array( [  [1,2,3] ]).transpose() #column vector

print("A =", A)

print("b =", b)

The simplest way to solve a linear system of equations is by calling the function "solve", which under the hood does all of the work for us:

In [ ]:
x = scipy.linalg.solve(A,b)
print("x = ", x)

If we have to solve many systems with matrix A and different b's, using the solve function every time is NOT the fastest option. 

Let's define matrix B, containing the RHS's we need to solve for:

In [ ]:
nrhs = 5
B = numpy.random.rand(A.shape[1], nrhs) #random matrix of size 3*nrhs
print(B)

The straightforward approach (aside from calling x = scipy.linalg.solve(A,B)) would be to call the solve function once per column:

In [ ]:
X = np.zeros(B.shape)
for i in range(B.shape[1]):
    b = B[:,i]
    X[:,i] = scipy.linalg.solve(A,b)

X2 = scipy.linalg.solve(A,B)
print(X)
print(X2)

We don't take advantage of the matrix A being the same for all systems. 
Factorizing the matrix is the most expensive operation. We should then do it only once, since A is only constant.

One option could be to first compute the inverse:

In [ ]:
Ainv = scipy.linalg.inv(A)
Xinv = Ainv@B

error = Xinv - X
print("error norm = ", scipy.linalg.norm(error))


Although apparently very effective, the computation of the inverse is **very expensive**.

Matrix inversion essentially consists in solving $n$ systems with matrix A, that come from imposing:
A @ Ainv = Identity => A @ Ainv[:,i] = Identity[:,i]

This is best understood by doing it:

In [ ]:
I = np.eye(A.shape[0]) #identity
our_inverse = np.zeros(A.shape)

our_inverse[:,0] = scipy.linalg.solve(A,I[:,0]) #first col
our_inverse[:,1] = scipy.linalg.solve(A,I[:,1]) #second col
our_inverse[:,2] = scipy.linalg.solve(A,I[:,2]) #third col

print("our_inverse = ", our_inverse)
print("inv(A) = ", scipy.linalg.inv(A))

#the original problem can now be solved as
print(our_inverse@B)
print("error norm = ", scipy.linalg.norm(our_inverse@B-X))

The bottomline here is that if the number of rows increases the computation of the inverse becomes very costly.
As a matter of fact **MATRIX INVERSION SHOULD BE ALMOST ALWAYS AVOIDED**.

The way to go here is to perform a LU decomposition:

In [ ]:
lu, piv = scipy.linalg.lu_factor(A) #LU contains the lower and upper triangular matrices and piv is the permutation vector
x = scipy.linalg.lu_solve((lu, piv), b)

print(x)

The original problem can be solved EFFECTIVELY as

In [ ]:
lu, piv = scipy.linalg.lu_factor(A)
X_lu = np.zeros(B.shape)
for i in range(B.shape[1]):
    b = B[:,i]
    X_lu[:,i] = scipy.linalg.lu_solve((lu, piv), b)
print("error norm = ",scipy.linalg.norm(X_lu-X))

LU decomposition (with pivoting) is possible for any invertible matrix. 

Let's see with an example why pivoting is needed sometimes:

In [ ]:
A = np.array([[1., -1, 2],[0,0,-1], [0,2,-1]])
b = np.array([[1],[2],[3]])

print("A = ",A)
print("b = ",b)

#the system is solvable,  with a solution
print("solution =", scipy.linalg.solve(A,b))

#but let's try to do it manually by the CROUT method:
L = np.zeros((3,3))
U = np.eye(3,3)

Crout factorization reads

$$
\mathbf{\left(\begin{array}{ccc}
a_{00} & a_{01} & a_{02}\\
a_{10} & a_{11} & a_{12}\\
a_{20} & a_{21} & a_{22}
\end{array}\right)=\left(\begin{array}{ccc}
l_{00} & 0 & 0\\
l_{10} & l_{11} & 0\\
l_{20} & l_{21} & l_{22}
\end{array}\right)\left(\begin{array}{ccc}
1 & u_{01} & u_{02}\\
0 & 1 & u_{12}\\
0 & 0 & 1
\end{array}\right)=\left(\begin{array}{ccc}
l_{00} & u_{0\text{1}}l_{00} & u_{02}l_{00}\\
l_{10} & l_{10}u_{01}+l_{11} & l_{10}u_{02}+l_{11}u_{12}\\
l_{20} & l_{20}u_{01}+l_{21} & l_{20}u_{02}+l_{21}u_{12}+l_{22}
\end{array}\right)}
$$

*   Step 1: $$a_{00} = l_{00} · 1$$





In [ ]:
L[0,0] = A[0,0]
print("step1 - L =\n",L)
print("step1 - U =\n",U)

*  Step 2: $$
\mathbf{\left(\begin{array}{cc}
a_{00} & a_{01} \\
a_{10} & a_{11} 
\end{array}\right)=\left(\begin{array}{cc}
l_{00} & 0 \\
l_{10} & l_{11} 
\end{array}\right)\left(\begin{array}{cc}
1 & u_{01} \\
0 & 1 
\end{array}\right)=\left(\begin{array}{ccc}
l_{00} & u_{0\text{1}}l_{00} \\
l_{10} & l_{10}u_{01}+l_{11} 
\end{array}\right)}
$$

In [ ]:
U[0,1] = A[0,1]/L[0,0]
L[1,0] = A[1,0]
L[1:2,1] = A[1:2,1] - L[1:2,0]*U[0,1]
print("step2 - L =\n",L) #note that L11 is zero!!
print("step2 - U =\n",U)

*   Step 3: $$
\mathbf{\left(\begin{array}{ccc}
a_{00} & a_{01} & a_{02}\\
a_{10} & a_{11} & a_{12}\\
a_{20} & a_{21} & a_{22}
\end{array}\right)=\left(\begin{array}{ccc}
l_{00} & 0 & 0\\
l_{10} & l_{11} & 0\\
l_{20} & l_{21} & l_{22}
\end{array}\right)\left(\begin{array}{ccc}
1 & u_{01} & u_{02}\\
0 & 1 & u_{12}\\
0 & 0 & 1
\end{array}\right)=\left(\begin{array}{ccc}
l_{00} & u_{0\text{1}}l_{00} & u_{02}l_{00}\\
l_{10} & l_{10}u_{01}+l_{11} & l_{10}u_{02}+l_{11}u_{12}\\
l_{20} & l_{20}u_{01}+l_{21} & l_{20}u_{02}+l_{21}u_{12}+l_{22}
\end{array}\right)}
$$


In [ ]:
U[0,2] = A[0,2]/L[0,0]
L[2,0] = A[2,0]
U[1,2] = (A[1,2] - U[0,2]*L[2,0])/L[1,1] 

#bum! HERE THE ALGORITHM FAILS!

What can we do? We can CHANGE THE ORDER OF THE OPERATIONS. (**PIVOTING**)


In [ ]:
#In fact let's consider that we have a matrix
P = np.array([
    [1,0,0],
    [0,0,1],
    [0,1,0]])

The solution of A@x = b must be the same as the solution of P@A@x = P@b (det(P)!= 0).

Let's try to compute the Crout factorization of P@A :

In [ ]:
L = np.zeros((3,3))
U = np.eye(3,3)

PA = P@A
Pb = P@b


# Step 1 
L[0,0] = PA[0,0]

# Step 2
U[0,1] = PA[0,1]/L[0,0]
L[1,0] = PA[1,0]
L[1:2,1] = PA[1:2,1] - L[1:2,0]*U[0,1]

# Step 3
U[0,2] = PA[0,2]/L[0,0]
L[2,0] = PA[2,0]
U[1,2] = (PA[1,2] - U[0,2]*L[2,0])/L[1,1]  #NOW IT WORKS!
L[2,2] = PA[2,2] - U[0,2]*L[2,0] - U[1,2]*L[2,1]

print("step 3 - L =\n",L) 
print("step 3 - U =\n",U)


We can confirm that L@U is the same as PA by taking the norm of the difference:

In [ ]:
print("error norm = ",scipy.linalg.norm(L@U-PA)) 

We can now solve L@U@x = Pb as 

1.   L@y = Pb
2.   U@x = y



In [ ]:
y = scipy.linalg.solve_triangular(L,Pb,lower=True)
x = scipy.linalg.solve_triangular(U,y,lower=False)
print("solution = ",x)
#same solution as before 

## **SYSTEM CONDITIONING**

We will see *system conditioning* as a measure of the numerical well-posedness of a given linear problem.

The condition number associated with $\mathbf{Ax} = \mathbf{b}$ gives a bound for the accuracy of the solution $\mathbf{x}$. Conditioning is a property of the matrix, not the algorithm or the floating-point accuracy of the computer. 

The condition number is (very roughly) the rate at which solution $\mathbf{x}$ will change with respect to a change of $\mathbf{b}$. 
*   If the condition number is large, even a small error in $\mathbf{b}$ may cause a large error in $\mathbf{x}$.
*   If the condition number is small, the error in $\mathbf{x}$ will not be much bigger than the error in $\mathbf{b}$.


In [ ]:
eb = 1.e-14 #perturbation on b

A = np.array([[1,1],[1,1+1.e-12]])
b = np.array([2,2])
bperturbed = np.array([2,2+eb])

print("Condition number A = ", numpy.linalg.cond(A))

x = np.linalg.solve(A,b)
xperturbed = np.linalg.solve(A,bperturbed)

print(" A x = b  --> ", x)
print(" A xperturbed = b --> ", xperturbed)

print("|b-bperturbed|/|b|--> ",scipy.linalg.norm(b-bperturbed)/scipy.linalg.norm(b))
print("|x-xperturbed|/|x|--> ",scipy.linalg.norm(x-xperturbed)/scipy.linalg.norm(x))

## **LEAST-SQUARES**

Let's now focus on the solution of __Least-Square problems__ of the type 
$$Ax = b,$$
where __A is a rectangular matrix with more rows that columns__ .

Since A is rectangular, the problem has no exact solution. We seek instead for the value "x" which minimizes
$$\Psi := (b-Ax)\cdot(b-Ax)= (b-Ax)^t(b-Ax).$$

The solution can be found by solving the normal equations:
$$A^tAx=A^tb.$$

In [ ]:
A = np.array([
        [1., 2.],
        [3., 4.],
        [3.,3.],
        [7.,8.]
    ])
b = np.array([
        [1.0],
        [3.],
        [3.],
        [5.]
    ])

AtA = A.transpose()@A
Atb = A.transpose()@b

print("Least Square solution = ", scipy.linalg.solve(AtA,Atb))

An interesting alternative is to employ to the same end the QR decomposition.

$$Ax=b \implies QRx=b \implies Q^tQRx=Q^tb \implies Rx=Q^tb$$

which delivers the same solution as $A^tAx=A^tb$ (easy proof, just do the substitution $A = QR$)

In [ ]:
Q,R = scipy.linalg.qr(A,mode="economic")
x = scipy.linalg.solve_triangular(R,Q.transpose()@b)
print("QR solution =", x)

Most interestingly the solution of the problem by QR has better numerical properties than $A^tAx=A^tb$:




In [ ]:
#try reducing this number all the way to e-10 (as we reduce it we make the two columns almost linearly dependent)
eps = 1e-10

A = np.array([
        [1., 1.+eps],
        [2., 2.],
        [3.,3.],
        [7.,7.]
    ])
b = np.array([
        [1.0],
        [3.],
        [3.],
        [5.]
    ])

#least square approach
xAtA = scipy.linalg.solve(A.transpose()@A,A.transpose()@b)
print("AtAx=Atb --> ", xAtA)

In [ ]:
Q,R = scipy.linalg.qr(A,mode="economic")
xQR = scipy.linalg.solve_triangular(R,Q.transpose()@b)
print("QRx=b --> ", xQR )

## **SVD DECOMPOSITION**

In [ ]:
A = np.array(
    [[0.28597345, 0.41441742, 0.56499227, 0.37924257, 0.16665891, 0.68519494],
     [0.17866702, 0.25934469, 0.35164773, 0.24094951, 0.10342719, 0.42823104],
     [0.3325819,  0.47702318, 0.6586028,  0.43081013, 0.19685866, 0.7966831 ]])

U,S,V = scipy.linalg.svd(A, full_matrices=False)

print("U\n",U)
print("S (it is a diagonal matrix)\n",S)
print("V\n",V)

__USE OF SVD IN APPROXIMATING MATRICES__:

Note that $\sigma_2 < \sigma_1 << \sigma_0$.
This is telling that the *modes* associated to $1$ and $2$ are less important than the ones associated to $0$. We can approximate the matrix by taking out the effect of the less important modes:

In [ ]:
k = 3
Uapprox = U[:,np.arange(k)].reshape(3,k) ##only the first k columns!
Sapprox = np.diag(S[np.arange(k)]) #only the first k diagonal entries (the others are approx 0)
Vapprox = V[np.arange(k),:].reshape(k,V.shape[1]) #only the first k rows!

print(Uapprox)
print(Sapprox) 
print(Vapprox)

Aapprox = Uapprox@Sapprox@Vapprox

print("A original =\n",A)
print("A approx =\n",Aapprox)
print("A - Aapprox = \n",A-Aapprox)
print("norm A-Approx = \n", np.linalg.norm(A-Aapprox)) #just to tell that A and Aapprox are very similar!

What we show just above is the basis of **Reduced Order Models**.